# dARK Lima five-host: acceptance of a deposit lifecycle

This notebook accepts the distributed Lima scenario: five independent Docker hosts, SSH management, four QBFT validators across two failure domains, and two storage peers on distinct machines. It exercises the public gateway on the `apps` VM and performs private Admin and Store checks through SSH without publishing additional ports.

It creates an authority and an ARK. Run it only in the disposable Lima lab after `install` and `verify`.


In [ ]:
from __future__ import annotations

import json, os, shlex, subprocess, sys, time, uuid
from pathlib import Path
from pprint import pprint

import requests

def find_repository_root() -> Path:
    configured = os.getenv('DARK_DEPLOYER_ROOT')
    candidates = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path.home() / 'source' / 'dark-deployer']
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'deploy.py').is_file() and (candidate / 'deployment_v3').is_dir():
            return candidate
    raise RuntimeError('Cannot find dark-deployer. Set DARK_DEPLOYER_ROOT to its absolute path.')

ROOT = find_repository_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INVENTORY = ROOT / os.getenv('DARK_INVENTORY', 'examples/operator-inventory/lima-five-host.json')
GATEWAY_BASE_URL = os.getenv('DARK_GATEWAY_BASE_URL', 'http://192.168.105.9').rstrip('/')
AUTHORITY_ID = os.getenv('AUTHORITY_ID', f'lima-notebook-{uuid.uuid4().hex[:12]}')
NAAN = os.getenv('NAAN', '12345').strip()
PLATFORM_ITEM_ID = os.getenv('PLATFORM_ITEM_ID', f'lima-item-{uuid.uuid4().hex[:12]}')
TARGET_URL = os.getenv('TARGET_URL', f'https://repository.example.org/items/{PLATFORM_ITEM_ID}')
POLL_INTERVAL_SECONDS = float(os.getenv('POLL_INTERVAL_SECONDS', '2'))
POLL_TIMEOUT_SECONDS = int(os.getenv('POLL_TIMEOUT_SECONDS', '240'))

print('inventory:', INVENTORY)
print('gateway:', GATEWAY_BASE_URL)
print('authority:', AUTHORITY_ID)


## 1. Resolve the distributed topology


In [ ]:
from deployment_v3.inventory_resolver import resolve_inventory_path
from deployment_v3.planner import build_plan
from deployment_v3.availability import analyze
from deployment_v3.network import internal_port

resolution = resolve_inventory_path(INVENTORY)
resolved = resolution.document
plan = build_plan(INVENTORY)
availability = analyze(plan)

assert resolved['deployment']['id'] == 'dark-lima-five-host'
assert resolved['images']['kubo']
assert availability.validators == 4 and availability.quorum == 3
assert availability.validator_failures_tolerated == 1
assert availability.storage_peers == 2
assert availability.storage_target_replicas == 2
assert len(availability.validator_machines) == 2
assert len({plan.service('ipfs-storage-a').machine_id, plan.service('ipfs-storage-b').machine_id}) == 2
print('catalog:', resolution.metadata['catalog'])
print('validators/quorum:', availability.validators, availability.quorum)
print('validator machines:', sorted(availability.validator_machines))
print('storage peers/target:', availability.storage_peers, availability.storage_target_replicas)


## 2. Verify public gateway and private SSH access

Admin and Store remain private. The helper executes the HTTP request inside the service container on the machine declared by the resolved topology.


In [ ]:
services = resolved['services']
gateway_id, gateway = next((service_id, service) for service_id, service in services.items() if service['type'] == 'edge-proxy')
assert gateway['exposure']['mode'] == 'public'
assert gateway['exposure']['port'] == 80
assert not services['admin-api'].get('exposure')
assert not services['store-api'].get('exposure')

MINTER_API_V1 = f'{GATEWAY_BASE_URL}/api/v1'
RESOLVER_ARK_BASE = f'{GATEWAY_BASE_URL}/'
MINTER_HEADERS = {'X-Authority-Id': AUTHORITY_ID}

def ssh_command(machine_id: str) -> list[str]:
    machine = resolved['machines'][machine_id]
    ssh = machine['ssh']
    return ['ssh', '-o', 'BatchMode=yes', '-o', 'StrictHostKeyChecking=yes', '-o', f"UserKnownHostsFile={ssh['known_hosts_file']}", '-i', ssh['private_key_file'], '-p', str(ssh['port']), f"{ssh['user']}@{machine['management_address']}"]

def remote_container_id(service_id: str) -> str:
    service = plan.service(service_id)
    group_id = next(group.id for group in plan.groups if service_id in group.service_ids)
    project = f'{plan.deployment_id}-{service.machine_id}-{group_id}'
    command = ['docker', 'ps', '--filter', f'label=com.docker.compose.project={project}', '--filter', f'label=com.docker.compose.service={service_id}', '--format', '{{.ID}}']
    result = subprocess.run([*ssh_command(service.machine_id), *command], text=True, capture_output=True, check=True)
    matches = [line for line in result.stdout.splitlines() if line]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one running {service_id} container in {project}; found {matches or result.stderr}')
    return matches[0]

REQUEST_SCRIPT = '''import json, sys, urllib.error, urllib.request
spec = json.load(sys.stdin)
body = None if spec['payload'] is None else json.dumps(spec['payload']).encode()
request = urllib.request.Request(spec['url'], data=body, method=spec['method'])
if body is not None: request.add_header('Content-Type', 'application/json')
try:
    response = urllib.request.urlopen(request, timeout=30); status, text = response.status, response.read().decode()
except urllib.error.HTTPError as error:
    status, text = error.code, error.read().decode()
print(json.dumps({'status': status, 'body': text}))'''

def private_request(service_id: str, method: str, path: str, *, payload: dict | None = None):
    service = plan.service(service_id)
    spec = {'method': method, 'url': f"http://127.0.0.1:{internal_port(service)}{path}", 'payload': payload}
    remote_command = ' '.join(shlex.quote(part) for part in ['docker', 'exec', '-i', remote_container_id(service_id), 'python', '-c', REQUEST_SCRIPT])
    result = subprocess.run([*ssh_command(service.machine_id), remote_command], input=json.dumps(spec), text=True, capture_output=True, check=True)
    response = json.loads(result.stdout)
    try: body = json.loads(response['body'])
    except json.JSONDecodeError: body = response['body']
    return response['status'], body

worker_status = requests.get(f'{MINTER_API_V1}/worker/status', timeout=30)
worker_status.raise_for_status()
for worker_name in ('metadata', 'replication', 'chain'):
    assert worker_status.json()['workers'][worker_name]['alive'] is True
assert private_request('admin-api', 'GET', '/health')[0] == 200
assert private_request('store-api', 'GET', '/health')[0] == 200
print('gateway, workers, Admin and Store are healthy')


## 3. Provision, publish, replicate, and resolve


In [ ]:
status, body = private_request('admin-api', 'POST', '/api/v1/admin/authority', payload={'uuid': AUTHORITY_ID, 'naans': [], 'fund_amount_eth': 0.05})
assert status in {200, 201, 409}, body
status, body = private_request('admin-api', 'POST', f'/api/v1/admin/authority/{AUTHORITY_ID}/authorize-naan', payload={'naan': NAAN})
assert status == 200, body

reserve_payload = {'authority_id': AUTHORITY_ID, 'naan': NAAN, 'items': [{'client_item_id': PLATFORM_ITEM_ID}]}
reserve = requests.post(f'{MINTER_API_V1}/arks/batch', headers=MINTER_HEADERS, json=reserve_payload, timeout=60)
reserve.raise_for_status(); reservation = reserve.json()['results'][0]
assert reservation['state'] == 'R', reservation
ARK = reservation['ark']
retry = requests.post(f'{MINTER_API_V1}/arks/batch', headers=MINTER_HEADERS, json=reserve_payload, timeout=60)
retry.raise_for_status(); assert retry.json()['results'][0]['ark'] == ARK

title = 'Objeto de aceptación Lima five-host'
level1 = {'title': title, 'authors': ['Equipo dARK'], 'year': 2026, 'publisher': 'Repositorio de demostración', 'resource_type': 'article', 'language': 'es', 'abstract': 'Registro creado por la aceptación distribuida Lima.', 'subjects': ['interoperabilidad', 'aceptación'], 'alternate_identifiers': [{'schema': 'platform-item-id', 'value': PLATFORM_ITEM_ID}], 'alternate_urls': [TARGET_URL]}
level2 = f'<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/" xmlns:dc="http://purl.org/dc/elements/1.1/"><dc:identifier>{TARGET_URL}</dc:identifier><dc:title>{title}</dc:title></oai_dc:dc>'
stage = requests.put(f'{MINTER_API_V1}/arks/{ARK}', headers=MINTER_HEADERS, json={'authority_id': AUTHORITY_ID, 'target': TARGET_URL, 'minimal_metadata': level1, 'original_metadata': level2, 'metadata_schema': 'oai_dc', 'metadata_media_type': 'application/xml'}, timeout=60)
stage.raise_for_status(); assert stage.json()['state'] == 'D', stage.json()

def get_ark():
    response = requests.get(f'{MINTER_API_V1}/arks/{ARK}', headers=MINTER_HEADERS, timeout=30); response.raise_for_status(); return response.json()
deadline = time.time() + POLL_TIMEOUT_SECONDS
while time.time() < deadline:
    published = get_ark(); print(time.strftime('%H:%M:%S'), published['state'])
    if published['state'] == 'P': break
    time.sleep(POLL_INTERVAL_SECONDS)
else: raise TimeoutError(f'ARK {ARK} did not reach PUBLISHED: {published}')
assert published['level1_cid'] and published['level2_cid']
print('published:', ARK)


In [ ]:
def wait_for_replication(cid: str):
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    while time.time() < deadline:
        status, payload = private_request('store-api', 'GET', f'/v1/status/{cid}')
        assert status == 200, payload
        replication = payload.get('replication', {})
        print(cid, replication)
        if replication.get('error_replicas', 0): raise RuntimeError(payload)
        if replication.get('total_replicas', 0) >= availability.storage_target_replicas: return payload
        time.sleep(POLL_INTERVAL_SECONDS)
    raise TimeoutError(f'CID {cid} did not reach {availability.storage_target_replicas} replicas')

for cid in (published['level1_cid'], published['level2_cid']):
    wait_for_replication(cid)
redirect = requests.get(f'{RESOLVER_ARK_BASE}{ARK}', allow_redirects=False, timeout=30)
assert redirect.status_code in {302, 307}, (redirect.status_code, redirect.text)
assert redirect.headers['location'] == TARGET_URL
print(f'Acceptance passed: {PLATFORM_ITEM_ID} -> {ARK} -> {TARGET_URL}')
print('Topology: 5 hosts; 4 validators across 2 machines; 2 IPFS copies across 2 storage machines.')
